# DuckDB + SLayer, from the command line

This shows how to use SLayer with DuckDB entirely from the command line, from scratch. 

We install the DuckDB CLI, expose a **48 KB CSV on a CDN** as a DuckDB view, let SLayer **auto-ingest** its schema into a semantic model, and query it with `slayer query`. Nothing is copied locally — the view points at the URL and every query reaches back over the wire.

Each `slayer query` runs in a shell cell, outputting results as json. For readability, we use pandas to display it.

**Prerequisites:** `pip install motley-slayer`.

## 1. Set it up — all CLI

Three steps, no Python: install the DuckDB CLI, create a **view** over the remote CSV inside a DuckDB file, then point a SLayer datasource at that file and let `--ingest` **auto-build the model** by introspecting the view. Column names and types come straight from the data — nothing hand-written.

In [1]:
%%bash
set -euo pipefail
export SLAYER_STORAGE=.cache/cli/store
rm -rf .cache/cli && mkdir -p .cache/cli

# Install the DuckDB CLI if it isn't already on PATH (idempotent). The installer
# uses bash syntax, so pipe it to bash — /bin/sh is dash on many CI runners.
command -v duckdb >/dev/null 2>&1 || curl -fsSL https://install.duckdb.org | bash >/dev/null
export PATH="$HOME/.duckdb/cli/latest:$PATH"

# Expose the remote CSV as a DuckDB view: the CSV stays on the CDN, the view
# just points at it over httpfs — nothing is copied locally.
duckdb .cache/cli/weather.duckdb -c \
  "CREATE VIEW weather AS SELECT * FROM read_csv_auto('https://cdn.jsdelivr.net/npm/vega-datasets@2/data/seattle-weather.csv')"

# Register the datasource and auto-ingest the view into a semantic model —
# column names and types come from introspecting the view, nothing hand-written.
PYTHONWARNINGS=ignore slayer datasources create "duckdb:///$PWD/.cache/cli/weather.duckdb" \
  --name weather_db --ingest -y
slayer models list

Created datasource 'weather_db' (duckdb).
Ingested: weather (6 columns, 0 measures)


weather


## 2. A warm-up query

A plain grouped count — how many days of each weather type. The shell cell runs `slayer query ... --format json` and captures its output; the next line renders it with pandas.

In [2]:
%%bash --out warmup
export SLAYER_STORAGE=.cache/cli/store
slayer query '{"source_model": "weather", "dimensions": ["weather"], "measures": [{"formula": "*:count", "name": "days"}]}' --format json

In [3]:
import json

import pandas as pd
from IPython.display import Markdown

pd.DataFrame(json.loads(warmup))

,weather.weather,weather.days
0,drizzle,53
1,rain,641
2,sun,640
3,fog,101
4,snow,26


## 3. The hero query: an aggregate as a dimension, plus a ranking transform

One query, single stage, two query-time features at the month grain:

- **A dimension computed from an aggregate.** `season` is `CASE WHEN temp_max:avg(partition_by=date) >= 18 THEN 'warm' ELSE 'cool' END` — each month is grouped *warm* or *cool* by its own average high, a value that only exists after aggregating. SLayer computes it in a synthesized stage and regroups on it.
- **A ranking transform in a measure.** `rank(precipitation:sum)` orders the months by rainfall, 1 = wettest.

Ordered wettest-first, the answer tells a story: Seattle's rainiest months are all *cool*-season.

We write the query to a file and run it with `slayer query @file` — the same JSON the Python notebook hands to the client.

In [4]:
%%bash --out hero_rows
set -e
export SLAYER_STORAGE=.cache/cli/store
cat > .cache/cli/hero.json <<'JSON'
{
  "source_model": "weather",
  "time_dimensions": [{"dimension": "date", "granularity": "month"}],
  "dimensions": [
    {"expression": "CASE WHEN temp_max:avg(partition_by=date) >= 18 THEN 'warm' ELSE 'cool' END", "name": "season"}
  ],
  "measures": [
    {"formula": "precipitation:sum", "name": "total_rain"},
    {"formula": "rank(precipitation:sum)", "name": "rain_rank"}
  ],
  "order": [{"column": "rain_rank", "direction": "asc"}]
}
JSON
slayer query @.cache/cli/hero.json --format json

In [5]:
pd.DataFrame(json.loads(hero_rows))

,weather.season,weather.date,weather.total_rain,weather.rain_rank
0,cool,2015-12-01 00:00:00,284.5,1
1,cool,2014-03-01 00:00:00,240.0,2
2,cool,2015-11-01 00:00:00,212.6,3
3,cool,2012-11-01 00:00:00,210.5,4
4,cool,2012-03-01 00:00:00,183.0,5
5,cool,2012-12-01 00:00:00,174.0,6
6,cool,2012-01-01 00:00:00,173.3,7
7,cool,2014-10-01 00:00:00,171.5,8
8,cool,2012-10-01 00:00:00,170.3,9
9,warm,2013-09-01 00:00:00,156.8,10


## 4. The SQL SLayer generated

But why would you want to use SLayer at all, rather than call DuckDB directly using SQL? 

This cell shows why. Compare the simple, natural json query above and the SQL below - which of them is an agent more likely to write correctly, every time?

In [6]:
%%bash --out hero_sql
export SLAYER_STORAGE=.cache/cli/store
slayer query @.cache/cli/hero.json --dry-run

In [7]:
Markdown("```sql\n" + hero_sql + "\n```")

```sql
SELECT
    "weather.season",
    "weather.date",
    "weather.total_rain",
    "weather.rain_rank"
FROM (
WITH _cm_temp_max_avg_partition_by_date AS (
  SELECT
    _stage_inner."weather.date_month" AS "date_month",
    _stage_inner."weather.temp_max_avg_partition_by_date" AS "temp_max_avg_partition_by_date"
  FROM (
    SELECT
      DATE_TRUNC('MONTH', weather.date) AS "weather.date_month",
      CAST(AVG(weather.temp_max) AS DOUBLE) AS "weather.temp_max_avg_partition_by_date"
    FROM weather AS weather
    GROUP BY
      DATE_TRUNC('MONTH', weather.date)
  ) AS _stage_inner
), base AS (
  SELECT
    CASE
      WHEN _cm_temp_max_avg_partition_by_date."temp_max_avg_partition_by_date" >= 18
      THEN 'warm'
      ELSE 'cool'
    END AS "weather.season",
    DATE_TRUNC('MONTH', weather.date) AS "weather.date",
    CAST(SUM(weather.precipitation) AS DOUBLE) AS "weather.total_rain"
  FROM weather AS weather
  LEFT JOIN _cm_temp_max_avg_partition_by_date
    ON DATE_TRUNC('MONTH', weather.date) IS NOT DISTINCT FROM _cm_temp_max_avg_partition_by_date."date_month"
  GROUP BY
    CASE
      WHEN _cm_temp_max_avg_partition_by_date."temp_max_avg_partition_by_date" >= 18
      THEN 'warm'
      ELSE 'cool'
    END,
    DATE_TRUNC('MONTH', weather.date)
), step1 AS (
  SELECT
    "weather.season",
    "weather.date",
    "weather.total_rain",
    RANK() OVER (ORDER BY "weather.total_rain" DESC) AS "weather.rain_rank"
  FROM base
)
SELECT
  "weather.season",
  "weather.date",
  "weather.total_rain",
  "weather.rain_rank"
FROM step1
) AS _outer
ORDER BY
  "weather.rain_rank" ASC

```

---

The whole semantic layer over a file on the internet — a DuckDB view, auto-ingestion, and JSON queries, all from the shell. See the [Python notebook](duckdb_python_nb.ipynb) for the in-process library version, [aggregations](../07_aggregations/aggregations.md) for more information about SLayer query syntax, and [formulas](../../concepts/formulas.md) for the full transform vocabulary.